In [ ]:
import json5
import pandas as pd
import regex as re
from Bio import SeqIO
from copy import deepcopy

added = {}
discarded = set()

replaced_accs = set()
replaced_seqs = dict()
replaced_genes = dict()

with open('sarg.json', 'r') as f:
    for i, j in json5.load(f).items():
        if i != 'discarded':
            for k, l in j.items():
                if k == 'discarded':
                    discarded.update(l)
                if k == 'changed':
                    replaced_seqs.update({z: x for x, y in l.items() for z in y if isinstance(y, list)})
                    replaced_accs.update({z.split('|')[-1] for x, y in l.items() for z in y if isinstance(y, list)})
                    replaced_genes.update({x: y for x, y in l.items() if not isinstance(y, list)})
                if k == 'added':
                    for m, n in l.items():
                        added[m] = 'REF|' + i + '|' + n + '|' + m
        else:
            discarded.update(j)

In [ ]:
records = []
dups = set()

ncbi = pd.read_table('reference/ReferenceGeneCatalog.txt').fillna('NA')
ncbi = ncbi[(ncbi['subtype'] == 'AMR') & (ncbi['class'] != 'NONE')]
ncbi['id'] = 'NCBI|' + (ncbi['type'] + ':' + ncbi['subtype']).str.lower() + '|' + ncbi['class'].str.lower().str.replace(' ', '_') + '|' + ncbi['gene_family'].str.replace(' ', '_')
refseq = ncbi.set_index('refseq_protein_accession')['id'].to_dict()
genbank = ncbi.set_index('genbank_protein_accession')['id'].to_dict()

with open('reference/AMRProt.fa') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        record.id = record.id.split('|')[0]
        record.name = record.id.split('|')[0]
        if record.id in refseq or record.id in genbank:
            record.id = refseq.get(record.id) + '|' + record.id if record.id in refseq else genbank.get(record.id) + '|' + record.id
            record.seq = record.seq[:-1] if record.seq[-1] == '*' else record.seq
            if record.id not in discarded and record.seq not in dups:
                dups.add(record.seq)
                records.append(record)

In [ ]:
%%bash
seqkit sort -s reference/reference.fasta -o reference/reference.sort.fasta --quiet
mv reference/reference.sort.fasta reference/reference.fasta

In [ ]:
reference = []
seen = set()
with open('reference/reference.fasta') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        seen.add(record.id)
        if record.seq not in dups:
            dups.add(record.seq)
            if record.id in added:
                old_record = deepcopy(record)
                reference.append(old_record)
                record.id = added.get(record.id)
                records.append(record)
            else:
                print('unused:', record.id)
        else:
            if record.id in added:
                reference.append(record)

with open('reference/reference.fasta', 'w') as handle:
    SeqIO.write(reference, handle, "fasta")

for record in added:
    if record not in seen:
        print(f'todo: {record}')

with open('tmp/seq2source.fa', 'w') as output_handle:
    SeqIO.write(records, output_handle, 'fasta')

In [ ]:
records_renamed = []
for record in records:
    if 'REF|' in record.id:
        if record.id in replaced_seqs:
            record.id = replaced_seqs.get(record.id)
        else:
            record.id = '@'.join(record.id.split('|')[1:3])
    elif 'NCBI|' in record.id:
        if record.id in replaced_seqs:
            record.id = replaced_seqs.get(record.id)
        else:
            record.id = '@'.join(record.id.split('|')[2:4])

    gene, id = record.id.split('@')[-1], record.id
    if record.name not in replaced_accs:
        for k, v in replaced_genes.items():
            id = re.sub(k, v, id)

    if 'sp|' in record.name or 'tr|' in record.name or 'gb|' in record.name:
        record.name = record.name.split('|')[1]

    record.id = f"SARG|{id.replace('@', '|')}|{record.name}"
    record.description = ''
    if id.split('@')[-1] not in discarded and id.split('@')[-2] not in discarded and gene not in discarded:
        records_renamed.append(record)

In [ ]:
with open('tmp/seq.fa', 'w') as output_handle:
    SeqIO.write(records_renamed, output_handle, 'fasta')

In [ ]:
%%bash
seqkit sort -s tmp/seq.fa -o tmp/seq.sort.fa --quiet
mv tmp/seq.sort.fa tmp/seq.fa